<h1>We're going to make a recycling model that identifies the type of trash in the image</h1>

<h3>We're going to start with importing every library we'll need</h3>

In [ ]:
%pip install keras
%pip install opencv-python
%pip install numpy
%pip install pandas
%pip install tensorflow-cpu==2.10
%pip install tensorflow-directml-plugin
%pip install tensorflow-rocm



  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 15.6 MB/s eta 0:00:00
Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 3.2/3.2 MB 47.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 12.9/12.9 MB 67.2 MB/s eta 0:00:00
Using cached namex-0.1.0-py3-none-any.whl (5.9 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
   ---------------------------------------- 0.0/262.7 MB ? eta -:--:--
    --------------------------------------- 4.2/262.7 MB 22.9 MB/s eta 0:00:12
   - -------------------------------------- 9.4/262.7 MB 25.5 MB/s eta 0:00:10
   -- ------------------------------------- 15.7/262.7 MB 26.0 MB/s eta 0:00:10
   --- ------------------------------------ 21.0/262.7 MB 25.5 MB/s eta 0:00:10
   --- ------------------------------------ 26.2/262.7 MB 26.0 MB/s eta 0:00:10
   ---- ------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ---------------------------------------- 0.0/9.0 MB ? eta -:--:--
   --------------------------- ------------ 6.3/9.0 MB 35.1 MB/s eta 0:00:01
   ---------------------------------------- 9.0/9.0 MB 25.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow-rocm (from versions: none)
ERROR: No matching distribution found for tensorflow-rocm



  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)


: 

In [1]:
import tensorflow as tf
print("GPUs Available: ", tf.config.list_physical_devices)

: 

In [1]:

import numpy as np 
import pandas as pd 
import keras
import cv2
import tensorflow as tf
import os


import matplotlib.pyplot as plt 
%matplotlib inline

from keras.models import Sequential
from keras.layers import Dense , Activation, Dropout
from keras.optimizers import Adam ,RMSprop
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

# Trying to avoid unecessary warnings
pd.options.mode.chained_assignment = None
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



KeyboardInterrupt



<h3>Now we'll import our data and make an x (input) and y (output) list</h3>

In [39]:
trainDataPath = 'data/dataMixStandardizWithIt'
trashTypes = os.listdir(trainDataPath)

xData = []
yData = []
# yTest = np.array(yTest, dtype='float32')

print(trashTypes)

['shoes', 'cardboard', 'plastic', 'glass', 'biological', 'clothes', 'battery', 'paper', 'metal']


In [40]:
resizeX = 16
resizeY = 16
count = 0
listSection = [1]
for i in (range(len(trashTypes)-1)):
    listSection.append(0)
# print(listSection)


# --------------------- The structure will be that each x will contain all the image data
# --------------------- and each corrosponding y will be the label of x

for i in (trashTypes):
    print("Adding " + i)
        
    # Creates a list of every item in the differing recycling category
    pathForiPath = os.listdir(trainDataPath+"/"+i)
    for j in pathForiPath:

        # Creates the individual path to each item that will be analyzed
        pathToTrashItem = (trainDataPath+"/"+i+"/"+j)
        # Converts the image to a list of individual (blue, green, red) lists per pixels
        bgrImg = cv2.imread(pathToTrashItem)
        bgrImg = cv2.resize(bgrImg, (resizeX,resizeY),interpolation=cv2.INTER_AREA)
        # bgrItem = np.resize(bgrImg, [resizeX,resizeY,3]).astype('float32')

        for k in range(4):
            xData.append(bgrImg.copy())
            yData.append(listSection.copy())
            bgrImg = cv2.rotate(bgrImg, cv2.ROTATE_90_CLOCKWISE)
            # print("rotating {}".format(str(k)))
            count = count+1
        # print(listSection)
    # print(listSection)
    listSection.pop()
    listSection.insert(0,0)
print("Count is {}".format(str(count)))
# print(xData[0])

Adding shoes
Adding cardboard
Adding plastic
Adding glass
Adding biological
Adding clothes
Adding battery
Adding paper
Adding metal
Count is 56588


In [41]:
print(yData)

[[1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 

<h3>Now that we have the xData and yData, lets make training and testing sets</h3>

In [42]:
xTrain, xTest, yTrain, yTest = train_test_split(xData, yData, test_size=0.20, random_state=42)
xTrain = np.resize(xTrain, [len(xTrain), resizeX,resizeY,3]).astype('float32')
xTest = np.resize(xTest, [len(xTest), resizeX,resizeY,3]).astype('float32')

yTrain = np.array(yTrain)
yTest = np.array(yTest)

print(len(yTest))

11318


<h3>Now to create the model</h3>

In [43]:
from tensorflow.keras import layers, models, regularizers
# model is a 3-layer MLP with ReLU and dropout after each layer
batch_size = 64
hidden_units = 256
# dropout = 0.20
classifier = Sequential(name = "Recycle Bot")
model = models.Sequential()

model.add(layers.Conv2D(32, (resizeX,resizeY), padding='same',
                        input_shape=(resizeX,resizeY,3),
                        activation=tf.nn.leaky_relu))

model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(4, 4)))
model.add(layers.Dropout(0.5))

model.add(layers.Conv2D(32, (resizeX,resizeY), padding='same',activation='selu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(4, 4)))
model.add(layers.Dropout(0.4))

model.add(layers.Conv2D(32, (resizeX,resizeY), padding='same',activation='gelu'))
# model.add(layers.BatchNormalization())
# model.add(layers.MaxPooling2D(pool_size=(2, 2)))
model.add(layers.Dropout(0.4))


model.add(layers.Flatten())
model.add(layers.Dense(400, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(len(trashTypes), activation='softmax'))
model.compile(optimizer='adam', 
              loss = 'categorical_crossentropy',
              metrics = ['accuracy'])
classifier = model

/home/puffle/anaconda3/envs/myenv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
hist = classifier.fit(xTrain, yTrain, epochs=2000, batch_size=batch_size)

Epoch 1/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 110s 155ms/step - accuracy: 0.4098 - loss: 1.6188
Epoch 2/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 110s 155ms/step - accuracy: 0.4141 - loss: 1.5956
Epoch 3/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 89s 126ms/step - accuracy: 0.4236 - loss: 1.5751
Epoch 4/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 107s 151ms/step - accuracy: 0.4294 - loss: 1.5535
Epoch 5/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 102s 144ms/step - accuracy: 0.4354 - loss: 1.5438
Epoch 6/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 9303s 13s/step - accuracy: 0.4407 - loss: 1.5322
Epoch 7/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 119s 167ms/step - accuracy: 0.4458 - loss: 1.5193
Epoch 8/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 99s 140ms/step - accuracy: 0.4486 - loss: 1.5080
Epoch 9/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 103s 145ms/step - accuracy: 0.4529 - loss: 1.5020
Epoch 10/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 102s 144ms/step - accuracy: 0.4535 - loss: 1.4928
Epoch 11/2000
708/708 ━━━━━━━━━━━━━━━━━━━━ 102s 145ms/step - accuracy: 0.4591 - lo

In [ ]:
model.save("withModifyModel_all90degRotations.keras")

In [45]:
loss, accuracy = classifier.evaluate(xTest, yTest, batch_size=batch_size)
print("\nTest Accuracy: "+str(100*accuracy)+" \nRandom Select Odds: "+str(100*(1/(len(trashTypes))))+"% \nPercent better than Random: "+str(100*accuracy-(100*(1/(len(trashTypes))))))


177/177 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - accuracy: 0.4356 - loss: 1.5665

Test Accuracy: 43.55893135070801 
Random Select Odds: 11.11111111111111% 
Percent better than Random: 32.44782023959689


<h1>My Own Predictions</h1>

In [ ]:
def predict(path):
    pathToTrashItem = path
    # Converts the image to a list of individual (blue, green, red) lists per pixels
    bgrImg = cv2.imread(pathToTrashItem)
    bgrImg = cv2.resize(bgrImg, (resizeX,resizeY),interpolation=cv2.INTER_AREA)
    bgrImg = np.resize(bgrImg, [1, resizeX,resizeY,3]).astype('float32')
    # bgrImg = np.array(bgrImg)
    # xData = np.resize(xData, [resizeX,resizeY,3]).astype('float32')
    pred = classifier.predict(bgrImg)
    max = 0
    for i in range(len(pred[0])):
        if pred[0][i] >= max:
            # print('hi')
            max = pred[0][i]
    for i in range(len(pred[0])):
        if pred[0][i] == max:
            return(trashTypes[i])
    return(pred)


print(predict("data/myOwn/battery_1.jpg"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
battery
